# Inherited House Price Predictions

## Objectives
* Predict inherited house prices using the chosen model from previous notebook. 

## Inputs
* `house_prices_records.csv` data located at outputs/datasets/collection
* `inherited_houses.csv` data located at outputs/datasets/collection. This is the dataset on customer's inherited houses' characteristics.
* Saved model under outputs/models/house_price_pipeline.joblib

## Outputs
- Predicted sale prices for the inherited houses.

## Imports and Setup

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import joblib

from sklearn.base import BaseEstimator, TransformerMixin

## Change Working Directory

In [3]:
project_root = Path.cwd()
# Move up until we find the project root (identified by a folder/file as marker)
while not (project_root / ".git").exists():
    project_root = project_root.parent

os.chdir(project_root)

print("Working directory set to:", project_root)

Working directory set to: /Users/mehtap/Documents/GitHub/heritage-housing-issues


## Load Data

Load inherited house data

In [5]:
df_predict = pd.read_csv("outputs/datasets/collection/inherited_houses.csv")
df_predict.head()

,1stFlrSF,2ndFlrSF,BedroomAbvGr,BsmtExposure,BsmtFinSF1,BsmtFinType1,BsmtUnfSF,EnclosedPorch,GarageArea,GarageFinish,...,LotArea,LotFrontage,MasVnrArea,OpenPorchSF,OverallCond,OverallQual,TotalBsmtSF,WoodDeckSF,YearBuilt,YearRemodAdd
0,896,0,2,No,468.0,Rec,270.0,0,730.0,Unf,...,11622,80.0,0.0,0,6,5,882.0,140,1961,1961
1,1329,0,3,No,923.0,ALQ,406.0,0,312.0,Unf,...,14267,81.0,108.0,36,6,6,1329.0,393,1958,1958
2,928,701,3,No,791.0,GLQ,137.0,0,482.0,Fin,...,13830,74.0,0.0,34,5,5,928.0,212,1997,1998
3,926,678,3,No,602.0,GLQ,324.0,0,470.0,Fin,...,9978,78.0,20.0,36,6,6,926.0,360,1998,1998


Ensure the feature set is the same as the training data

In [26]:
print(df_predict.columns.tolist())

['1stFlrSF', '2ndFlrSF', 'BedroomAbvGr', 'BsmtExposure', 'BsmtFinSF1', 'BsmtFinType1', 'BsmtUnfSF', 'EnclosedPorch', 'GarageArea', 'GarageFinish', 'GarageYrBlt', 'GrLivArea', 'KitchenQual', 'LotArea', 'LotFrontage', 'MasVnrArea', 'OpenPorchSF', 'OverallCond', 'OverallQual', 'TotalBsmtSF', 'WoodDeckSF', 'YearBuilt', 'YearRemodAdd']


## Custom Feature Engineering Transformer

(to be saved under scr for automation after ensuring predictions work here)

In [12]:
class FeatureEngineerHPData(BaseEstimator, TransformerMixin):
    def __init__(self, lotarea_large_q=0.99, lotarea_small_q=0.01,
                 bedroom_impute_strategy="mean"):
        self.lotarea_large_q = lotarea_large_q
        self.lotarea_small_q = lotarea_small_q
        self.bedroom_impute_strategy = bedroom_impute_strategy

    def fit(self, X, y=None):
        X = X.copy()

        # LotArea thresholds learned from training data
        self.large_lot_threshold_ = X['LotArea'].quantile(self.lotarea_large_q)
        self.small_lot_threshold_ = X['LotArea'].quantile(self.lotarea_small_q)

        # Modes learned from training data
        self.bsmtfintype1_mode_ = X['BsmtFinType1'].mode(dropna=True)[0]
        self.bedroom_mode_ = X['BedroomAbvGr'].mode(dropna=True)[0]

        # Mean learned from training data
        self.bedroom_mean_ = X['BedroomAbvGr'].mean()

        if self.bedroom_impute_strategy == "mean":
            self.bedroom_fill_value_ = self.bedroom_mean_
        elif self.bedroom_impute_strategy == "mode":
            self.bedroom_fill_value_ = self.bedroom_mode_
        else:
            raise ValueError("bedroom_impute_strategy must be 'mean' or 'mode'")

        return self

    def transform(self, X):
        X = X.copy()


        # =============================
        # Create missingness indicators
        # =============================
        # Create missingness indicators before imputations
        X['MissingLotFrontage'] = X['LotFrontage'].isna().astype(int)
        X['MissingMasVnrArea'] = X['MasVnrArea'].isna().astype(int)
        X['MissingGarageYrBlt'] = X['GarageYrBlt'].isna().astype(int)
        X['MissingBedroomAbvGr'] = X['BedroomAbvGr'].isna().astype(int)
        # Create missingness indicator after imputations 
        X['MissingBsmtFinType1'] = 0

        # ===============================
        # Replace missing cells with zero
        # ===============================
        X['2ndFlrSF'] = X['2ndFlrSF'].fillna(0)
        X['LotFrontage'] = X['LotFrontage'].fillna(0)
        X['MasVnrArea'] = X['MasVnrArea'].fillna(0)
        X['GarageYrBlt'] = X['GarageYrBlt'].fillna(0)
        X['EnclosedPorch'] = X['EnclosedPorch'].fillna(0)
        X['WoodDeckSF'] = X['WoodDeckSF'].fillna(0)

        #============================
        # Impute missing BedroomAbvGr
        #============================
        X['BedroomAbvGr'] = X['BedroomAbvGr'].fillna(self.bedroom_fill_value_)

        # ======================================
        # Create binary house feature indicators
        # ======================================
        X['Has2ndFlr'] = (X['2ndFlrSF'] > 0).astype(int)
        X['HasExtraLivArea'] = (X['GrLivArea'] > (X['1stFlrSF'] + X['2ndFlrSF'])).astype(int)
        X['HasBasement'] = (X['TotalBsmtSF'] > 0).astype(int)
        X['HasBsmtFin'] = (X['BsmtFinSF1'] > 0).astype(int)
        X['HasBsmtUnf'] = (X['BsmtUnfSF'] > 0).astype(int)
        X['HasGarage'] = (X['GarageArea'] > 0).astype(int)
        X['HasMasVnr'] = (X['MasVnrArea'] > 0).astype(int)
        X['HasEnclosedPorch'] = (X['EnclosedPorch'] > 0).astype(int)
        X['HasOpenPorch'] = (X['OpenPorchSF'] > 0).astype(int)
        X['HasWoodDeck'] = (X['WoodDeckSF'] > 0).astype(int)
        X['BuiltPre1950'] = (X['YearBuilt'] < 1950).astype(int)

        # ===========================================
        # Create large/small lot size dummy variables
        # ===========================================
        X['HasLargeLotArea'] = (X['LotArea'] > self.large_lot_threshold_).astype(int)
        X['HasSmallLotArea'] = (X['LotArea'] < self.small_lot_threshold_).astype(int)

        # =================================
        # Impute missing basement variables
        # ================================= 
        X.loc[(X["BsmtExposure"].isna()) & (X["TotalBsmtSF"] == 0), "BsmtExposure"] = "No_basement"
        X.loc[(X["BsmtExposure"].isna()) & (X["TotalBsmtSF"] > 0), "BsmtExposure"] = "No"

        X.loc[(X['BsmtFinType1'].isna()) & (X['TotalBsmtSF'] == 0), 'BsmtFinType1'] = "No_basement"
        X.loc[((X['BsmtFinType1'].isna()) & (X['BsmtUnfSF'] > 0) & (X['BsmtFinSF1'] == 0)), 'BsmtFinType1'] = "Unf"

        X.loc[X['BsmtFinType1'].isna(), 'MissingBsmtFinType1'] = 1
        X.loc[X['BsmtFinType1'].isna(), 'BsmtFinType1'] = self.bsmtfintype1_mode_

        # ============================
        # Impute missing garage finish
        # ============================
        X.loc[(X['GarageFinish'].isna()) & (X['GarageArea'] == 0), 'GarageFinish'] = "No_garage"
        X.loc[(X['GarageFinish'].isna()) & (X['GarageArea'] > 0), 'GarageFinish'] = "Missing"

        # ==============================
        # Create new feature (numerical)
        # ==============================
        X["TotalPorchSF"] = X["EnclosedPorch"] + X["OpenPorchSF"]

        return X

## Predict Inherited House Logarithmic Prices

In [28]:
loaded_model = joblib.load("outputs/models/house_price_pipeline.joblib")

Ensure that the loaded_model includes the feature engineering pipelines

In [32]:
type(loaded_model)

sklearn.pipeline.Pipeline

In [34]:
loaded_model

Pipeline(steps=[('feature_engineering', FeatureEngineerHPData()),
                ('preprocessor',
                 ColumnTransformer(transformers=[('log',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('log',
                                                                   FunctionTransformer(feature_names_out='one-to-one',
                                                                                       func=<ufunc 'log'>))]),
                                                  ['1stFlrSF', 'GrLivArea',
                                                   'LotArea']),
                                                 ('log1p',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(stra...
                                                   'HasSmallLotArea',
                                                   'HasMasVnr',
                                                   'HasEnclosedPorch',
                                                   'HasOpenPorch',
                                                   'HasWoodDeck',
                                                   'BuiltPre1950']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['BsmtExposure',
                                                   'BsmtFinType1',
                                                   'GarageFinish',
                                                   'KitchenQual'])])),
                ('model',
                 RandomForestRegressor(n_estimators=200, random_state=34))])

In [29]:
log_price_predictions = loaded_model.predict(df_predict)
price_predictions = np.exp(log_price_predictions)

df_predict_results = df_predict.copy()
df_predict_results["PredictedSalePrice"] = price_predictions

df_predict_results

,1stFlrSF,2ndFlrSF,BedroomAbvGr,BsmtExposure,BsmtFinSF1,BsmtFinType1,BsmtUnfSF,EnclosedPorch,GarageArea,GarageFinish,...,LotFrontage,MasVnrArea,OpenPorchSF,OverallCond,OverallQual,TotalBsmtSF,WoodDeckSF,YearBuilt,YearRemodAdd,PredictedSalePrice
0,896,0,2,No,468.0,Rec,270.0,0,730.0,Unf,...,80.0,0.0,0,6,5,882.0,140,1961,1961,124407.053598
1,1329,0,3,No,923.0,ALQ,406.0,0,312.0,Unf,...,81.0,108.0,36,6,6,1329.0,393,1958,1958,157170.742599
2,928,701,3,No,791.0,GLQ,137.0,0,482.0,Fin,...,74.0,0.0,34,5,5,928.0,212,1997,1998,169290.139425
3,926,678,3,No,602.0,GLQ,324.0,0,470.0,Fin,...,78.0,20.0,36,6,6,926.0,360,1998,1998,185306.232258


Round to two decimal points

In [30]:
df_predict_results["PredictedSalePrice"] = df_predict_results["PredictedSalePrice"].round(2)
df_predict_results

,1stFlrSF,2ndFlrSF,BedroomAbvGr,BsmtExposure,BsmtFinSF1,BsmtFinType1,BsmtUnfSF,EnclosedPorch,GarageArea,GarageFinish,...,LotFrontage,MasVnrArea,OpenPorchSF,OverallCond,OverallQual,TotalBsmtSF,WoodDeckSF,YearBuilt,YearRemodAdd,PredictedSalePrice
0,896,0,2,No,468.0,Rec,270.0,0,730.0,Unf,...,80.0,0.0,0,6,5,882.0,140,1961,1961,124407.05
1,1329,0,3,No,923.0,ALQ,406.0,0,312.0,Unf,...,81.0,108.0,36,6,6,1329.0,393,1958,1958,157170.74
2,928,701,3,No,791.0,GLQ,137.0,0,482.0,Fin,...,74.0,0.0,34,5,5,928.0,212,1997,1998,169290.14
3,926,678,3,No,602.0,GLQ,324.0,0,470.0,Fin,...,78.0,20.0,36,6,6,926.0,360,1998,1998,185306.23


Save predictions

In [31]:
df_predict_results.to_csv("outputs/datasets/predictions/predicted_house_prices.csv", index=False)
print("Prediction file saved.")

Prediction file saved.


Sum of predicted prices for all four inherited houses:

In [35]:
df_predict_results["PredictedSalePrice"].sum()

636174.16

## Summary

- A machine learning pipeline is developed to predict house sale prices in Ames, Iowa.  
- The workflow automated feature engineering (new feature creation, missing value handling, transformations, categorical encoding, etc.) model fitting, and prediction.  
- Candidate models are compared using R2 score, RMSE and MAE. All alternative models have an R2 score above the client's suggested threshold of 0.75.      
- A random forest model is selected as the final model to be used in predictions.
- The selected pipeline is saved for deployment and used to predict prices for the four inherited houses.
- The total predicted value of these four houses is calculated to be 636,174.16.

The overall work done supports the business requirement of helping the customer estimate the value of their inherited houses and future Ames properties reliably.